# CNN — The Library Version

Three verifications, honestly framed: (1) our convolution against SciPy's, numerically; (2) our accuracy against a strong sklearn baseline; (3) the PyTorch translation — shown, not run, since the industrial deep-learning stack lives outside this environment. Nothing in it will be new to you.

In [1]:
import numpy as np
import pandas as pd
from scipy.signal import correlate2d

df = pd.read_csv("data/digits_data.csv")
y = df["label"].values
X = df.drop(columns="label").values.reshape(-1, 8, 8)

# (1) verify the core operation: our patches@K conv vs scipy's correlate2d
def patches3(imgs):
    n = imgs.shape[0]
    P = np.empty((n, 6, 6, 9))
    for i in range(6):
        for j in range(6):
            P[:, i, j, :] = imgs[:, i:i+3, j:j+3].reshape(n, 9)
    return P

rs = np.random.default_rng(0)
k = rs.normal(0, 1, (3, 3))
ours  = (patches3(X[:5]) @ k.ravel())
scipys = np.stack([correlate2d(img, k, mode="valid") for img in X[:5]])
print(f"our conv vs scipy.correlate2d — max |difference|: {np.abs(ours - scipys).max():.2e}")
print("(identical: 'convolution' in deep learning is cross-correlation; the patches@K trick is exact)")

our conv vs scipy.correlate2d — max |difference|: 4.44e-16
(identical: 'convolution' in deep learning is cross-correlation; the patches@K trick is exact)


In [2]:
# (2) the strong classical baseline on the same split
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

Xf = X.reshape(len(y), -1)
Xtr, Xte, ytr, yte = train_test_split(Xf, y, test_size=0.2, random_state=99)
mlp = MLPClassifier(hidden_layer_sizes=(64,), max_iter=600, random_state=0).fit(Xtr, ytr)
print(f"sklearn MLP (64 hidden): {accuracy_score(yte, mlp.predict(Xte)):.1%}")
print("Honest comparison: this 4,874-parameter MLP beats our 810-parameter scratch CNN")
print("(94.4%) by a few points on CLEAN data — 8x8 digits are small enough that brute dense")
print("capacity competes. The CNN's case here is efficiency (6x fewer parameters for within")
print("a few points) and robustness (it degrades less under shift — scratch notebook, Block 9).")
print("Architectural advantages grow with image size; at 8x8 they are visible, not dramatic.")

sklearn MLP (64 hidden): 98.3%
Honest comparison: this 4,874-parameter MLP beats our 810-parameter scratch CNN
(94.4%) by a few points on CLEAN data — 8x8 digits are small enough that brute dense
capacity competes. The CNN's case here is efficiency (6x fewer parameters for within
a few points) and robustness (it degrades less under shift — scratch notebook, Block 9).
Architectural advantages grow with image size; at 8x8 they are visible, not dramatic.


### (3) The PyTorch translation — read it; you have built every line

```python
import torch.nn as nn

model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3),   # Block 4: patches @ K  (+ autograd writes Block 7 for you)
    nn.ReLU(),                        # Block 5
    nn.MaxPool2d(2),                  # Block 5 (argmax routing handled in its backward)
    nn.Flatten(),
    nn.Linear(8*3*3, 10),             # Block 6's dense head
)                                      # CrossEntropyLoss = softmax + CE, fused (miracle included)
```

Every layer maps to a block you wrote by hand; `loss.backward()` is your Block 7, generated automatically for any architecture. That — plus GPUs — is the entire gap between this notebook and production vision models.